# ЛР 4. Осмысленные дескрипторы

**Сущность:** Посчитать признаки и отличить нулевое значение, пропуск и контекстное предупреждение.

**Данные:** Те же 12 строк; признаки рассчитываются только для валидного parent.

**Workflow:** Стандартизация → признаки с единицами → четыре условия Ro5 → объяснение.

Перед запуском прочитайте [guideline](../guides/lab04.md) и [теорию](../../Module_theory/Module%204_Descriptor.md).

## 1. Прогноз

Запишите ожидаемое поведение и отрицательный контроль. Какое изменение входа должно изменить результат, а какое — сохранить?

In [1]:
from pathlib import Path
import sys, json, math, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# WHY: relative discovery makes the notebook portable across machines.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Labs" / "chemistry.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите notebook внутри репозитория курса")
sys.path.insert(0, str(ROOT / "Labs"))
DATA = ROOT / "Labs" / "datasets"
OUT = ROOT / "Labs" / "results" / "lab04"
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()


## 2. Разбор и исполнение

1. Сравните этанол и бензол качественно до расчёта.
2. Выведите MW, MolLogP, TPSA, HBD, HBA и число предупреждений.
3. Проверьте полиол, для которого важны HBD/HBA, а не только масса и липофильность.

Ожидаемое: MW этанола 46,069±0,01; TPSA бензола 0; контрольный полиол нарушает HBD/HBA.

In [2]:
from chemistry import standardize, descriptors
raw=pd.read_csv(DATA/"molecules.csv",keep_default_na=False)
table=[]
for row in raw.itertuples():
    try:
        parent=standardize(row.smiles).parent_smiles
    except ValueError:
        continue
    table.append({"id":row.id,"parent":parent,**descriptors(parent)})
features=pd.DataFrame(table); display(features)
features.to_csv(OUT/"features.csv",index=False)
assert math.isclose(descriptors("CCO")["MolWt"],46.069,abs_tol=0.01)
assert descriptors("c1ccccc1")["TPSA"]==0
polyol=descriptors("OC(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)CO")
assert polyol["HBD"]>5 and polyol["HBA"]>10 and polyol["ro5_violations"]==2
artifact={"valid_rows":len(features),"ethanol":descriptors("CCO"),"polyol":polyol}


[18:22:26] Initializing MetalDisconnector
[18:22:26] Running MetalDisconnector
[18:22:26] Initializing Normalizer
[18:22:26] Running Normalizer
[18:22:26] Initializing MetalDisconnector
[18:22:26] Running MetalDisconnector
[18:22:26] Initializing Normalizer
[18:22:26] Running Normalizer
[18:22:26] Running LargestFragmentChooser
[18:22:26] Running Uncharger
[18:22:26] Initializing MetalDisconnector
[18:22:26] Running MetalDisconnector
[18:22:26] Initializing Normalizer
[18:22:26] Running Normalizer
[18:22:26] Initializing MetalDisconnector
[18:22:26] Running MetalDisconnector
[18:22:26] Initializing Normalizer
[18:22:26] Running Normalizer
[18:22:26] Running LargestFragmentChooser
[18:22:26] Running Uncharger
[18:22:26] Initializing MetalDisconnector
[18:22:26] Running MetalDisconnector
[18:22:26] Initializing Normalizer
[18:22:26] Running Normalizer
[18:22:26] Initializing MetalDisconnector
[18:22:26] Running MetalDisconnector
[18:22:26] Initializing Normalizer
[18:22:26] Running Norma

,id,parent,MolWt,MolLogP,TPSA,HBD,HBA,ro5_violations
0,M01,CCO,46.069,-0.0014,20.23,1,1,0
1,M02,CCO,46.069,-0.0014,20.23,1,1,0
2,M03,CC(=O)O,60.052,0.0909,37.30,1,1,0
3,M04,CC(=O)O,60.052,0.0909,37.30,1,1,0
4,M05,Clc1ccccc1,112.559,2.3400,0.00,0,0,0
5,M06,Oc1ccccc1O,110.112,1.0978,40.46,2,2,0
6,M07,c1ccccc1,78.114,1.6866,0.00,0,0,0
7,M10,C[C@H](O)C(=O)O,90.078,-0.5482,57.53,2,2,0
8,M11,C[C@@H](O)C(=O)O,90.078,-0.5482,57.53,2,2,0
9,M12,CCCC,58.124,1.8064,0.00,0,0,0


## 3. Проверяемая запись запуска

Benchmark фиксирует этот запуск. Время включает расчёт и создание артефактов, но не установку среды; это не сравнительный рейтинг оборудования.

In [3]:
elapsed = time.perf_counter()-started
run_record = {"lab": 4, "calculation_wall_seconds": elapsed, "result": artifact}
(OUT/"benchmark.json").write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding="utf-8")
print("Запуск сохранён:",OUT/"benchmark.json")


Запуск сохранён: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\Labs\results\lab04\benchmark.json


## 4. Самостоятельная тренировка

1. Восстановите формулу счётчика четырёх нарушений.
2. Найдите два признака, которые могут быть связаны, и объясните, почему это не две независимые причины активности.
3. Объясните отличие logP от logD(pH).

[Ответы после попытки](../solutions/lab04.md). Вернитесь к первому заданию через неделю.

## 5. Ваш научный вывод

1. Какой результат получен?
2. Какой вывод из него пока не следует?
3. Какой следующий эксперимент или контроль нужен?

Сохраните свой вариант notebook отдельно; исходный пример должен остаться воспроизводимым.